# Practical 12 --- A Pipeline You Can Re-run

**Tasks in the right order, failures that stop the line, and a schedule**

SCSE3040 Machine Learning Operations &middot; Bennett University &middot; Session 2026-27

| | |
|---|---|
| **Lectures this follows** | L24-L25 |
| **Course Outcome** | CO4 |
| **Lab duration** | 120 minutes |
| **Memory needed** | about 400 MB (fine on a 4 GB or 8 GB machine) |
| **Extra software** | nothing beyond the course venv |
| **Marks** | 10 |

---

## Aim

1. Break a job into small tasks and say which depends on which.
2. Work out a safe order to run them in, automatically.
3. Make a pipeline safe to re-run, and safe to fail half way through.
4. Read a cron schedule and say when a job will next run.

---

## What you need to know first


Everything you have built so far runs when you press a button. Real systems do
not work like that. Every night, without anyone awake, the same steps have to
happen in the same order: fetch yesterday's deliveries, check them, build the
features, retrain, write a report.

That chain is a **data pipeline**, and the tool that runs it in production is
usually **Apache Airflow**. Airflow needs a database, a scheduler, a web server
and a couple of gigabytes of memory --- more than a lab machine can spare. So
today you build the *ideas* Airflow implements, in about forty lines of Python.
Everything you learn transfers directly.

The central idea is the **DAG**, short for **Directed Acyclic Graph**. Take that
apart:

- **Graph** --- tasks with arrows between them
- **Directed** --- the arrows point one way: *train* comes after *validate*
- **Acyclic** --- no loops. If A waits for B and B waits for A, nothing can
  ever start.

From a DAG you can work out an order to run things in. That calculation is
called a **topological sort**, and you will write one.

Two properties separate a pipeline that survives from one that ruins a Sunday.

**Idempotent** means running it twice gives the same result as running it once.
Yesterday's run crashed half way; you fix the bug and run it again. If the
pipeline appends rather than replaces, you now have every row twice, and nobody
notices for a month.

**Fail fast** means when a task fails, everything downstream is *skipped*, not
attempted. If validation fails, training must not run --- a model trained on bad
data is worse than no model, because it looks fine.

Finally, **cron**: five fields saying when to run. `0 2 * * *` means "minute 0,
hour 2, every day". You will meet it in Airflow, in Kubernetes CronJobs, and on
every Linux server you ever touch.

---

## Before you start

- Practicals P04 and P05 are finished. You can write functions and tests.
- You know what a dictionary and a list are.

### How to run a cell

Click on a grey code cell, then press **Shift + Enter**. The cell runs and
the cursor moves to the next one. A number appears in the `[ ]` on the left
when the cell has finished.

**Run the cells in order, from the top.** A later cell almost always uses
something an earlier cell created. If you jump ahead you will see a
`NameError`, which just means "you have not made that thing yet".

If everything goes wrong, use the menu: **Kernel -> Restart Kernel and Clear
All Outputs**, then start again from the first cell. Nothing is damaged by
doing this.

In [ ]:
# Step 0 --- check the workbench before we start.
# This cell only looks; it changes nothing. Run it and read the last line.

import sys
from pathlib import Path

print("Python  :", sys.version.split()[0])
print("Folder  :", Path.cwd().name)

_missing = []
for _name in ['numpy', 'pandas', 'sklearn']:
    try:
        __import__(_name)
    except ImportError:
        _missing.append(_name)

for _name in ['numpy', 'pandas', 'sklearn']:
    _mark = "missing" if _name in _missing else "ok"
    print(f"  {_name:<14} {_mark}")

if _missing:
    print()
    print("STOP. Some libraries are missing:", ", ".join(_missing))
    print("Ask your instructor to run the setup in labs/SETUP.md.")
else:
    print()
    print("All good. You can carry on to Step 1.")

---

## Step 0b --- the dataset

Every practical in this course uses the same 600 food deliveries.
The next cell makes sure the file is there.

In [ ]:
# The delivery-time dataset every practical in this course uses.
# If the file is missing we build it again from the same seed, so every
# student in the room gets byte-for-byte the same 600 rows.

import csv
from pathlib import Path

import numpy as np

SEED = 42
N_ROWS = 600
DATA = Path("..") / "data" / "delivery_times.csv"


def make_delivery_csv(path=DATA):
    """Write the 600-row delivery dataset. Same formula as the lectures."""
    rng = np.random.default_rng(SEED)
    distance_km = np.round(rng.uniform(0.5, 12.0, N_ROWS), 2)
    prep_time_min = np.round(rng.uniform(5, 30, N_ROWS), 0)
    traffic_level = rng.integers(1, 4, N_ROWS)
    rain = rng.binomial(1, 0.25, N_ROWS)
    delivery_min = np.round(
        6.0
        + 3.1 * distance_km
        + 0.65 * prep_time_min
        + 4.2 * traffic_level
        + 5.5 * rain
        + rng.normal(0, 2.5, N_ROWS),
        1,
    )
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as fh:
        w = csv.writer(fh)
        w.writerow(["distance_km", "prep_time_min", "traffic_level",
                    "rain", "delivery_min"])
        for i in range(N_ROWS):
            w.writerow([distance_km[i], int(prep_time_min[i]),
                        int(traffic_level[i]), int(rain[i]), delivery_min[i]])
    return path


if not DATA.exists():
    make_delivery_csv()
    print("dataset rebuilt ->", DATA)
else:
    print("dataset found   ->", DATA)

---

# Walkthrough

Read each step, then run its cell.

### Step 1 --- Write the tasks as ordinary functions

A pipeline is not a special kind of code. It is ordinary functions,
each doing one job, plus a description of the order.

Each task takes a shared `context` dictionary and adds to it. That
is exactly how Airflow passes data between tasks.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

WORK = Path("work")
WORK.mkdir(exist_ok=True)
FEATURES = ["distance_km", "prep_time_min", "traffic_level", "rain"]


def extract(context):
    """Read yesterday's deliveries."""
    context["orders"] = pd.read_csv(DATA)
    return f"read {len(context['orders'])} orders"


def validate(context):
    """Refuse to continue on obviously broken data."""
    orders = context["orders"]
    problems = []
    if orders["distance_km"].min() <= 0:
        problems.append("a distance of zero or less")
    if orders["delivery_min"].isna().any():
        problems.append("missing delivery times")
    if not orders["traffic_level"].isin([1, 2, 3]).all():
        problems.append("a traffic level outside 1-3")
    if problems:
        raise ValueError("; ".join(problems))
    return f"{len(orders)} rows look sane"


def transform(context):
    """Build the model's inputs."""
    orders = context["orders"]
    context["X"] = orders[FEATURES]
    context["y"] = orders["delivery_min"]
    return f"{context['X'].shape[1]} features prepared"


def train(context):
    """Fit the model and remember its error."""
    X_tr, X_te, y_tr, y_te = train_test_split(
        context["X"], context["y"], test_size=0.2, random_state=42
    )
    model = LinearRegression().fit(X_tr, y_tr)
    context["model"] = model
    context["mae"] = mean_absolute_error(y_te, model.predict(X_te))
    return f"MAE {context['mae']:.2f} minutes"


def report(context):
    """Write the one file a human will actually look at."""
    path = WORK / "daily_report.txt"
    path.write_text(
        f"rows: {len(context['orders'])}\n"
        f"mae_minutes: {context['mae']:.3f}\n",
        encoding="utf-8",
    )
    return f"wrote {path.name}"


print("five tasks defined")

### Step 2 --- Describe the order as a graph

Now say what depends on what. Read this dictionary as "this task
waits for these tasks".

Notice `report` waits for `train`, and `train` waits for
`transform`. Nothing says *when* anything runs --- only what must
come first.

In [ ]:
DAG = {
    "extract":   [],
    "validate":  ["extract"],
    "transform": ["validate"],
    "train":     ["transform"],
    "report":    ["train"],
}

TASKS = {
    "extract": extract,
    "validate": validate,
    "transform": transform,
    "train": train,
    "report": report,
}

for name, waits_for in DAG.items():
    arrow = " <- " + ", ".join(waits_for) if waits_for else ""
    print(f"  {name}{arrow}")

### Step 3 --- Work out a safe order

A **topological sort**: repeatedly take any task whose dependencies
are all done. If at some point nothing is ready but tasks remain,
there is a loop, and we say so rather than hanging forever.

In [ ]:
def plan(dag):
    """Return an order in which every task's dependencies come first."""
    order, done = [], set()
    remaining = dict(dag)
    while remaining:
        ready = [name for name, deps in remaining.items()
                 if all(d in done for d in deps)]
        if not ready:
            raise ValueError(
                "these tasks depend on each other in a loop: "
                + ", ".join(sorted(remaining))
            )
        for name in sorted(ready):        # sorted = same order always
            order.append(name)
            done.add(name)
            del remaining[name]
    return order


ORDER = plan(DAG)
print("run order:", " -> ".join(ORDER))

Sorting the ready list matters more than it looks. Without it, the
order could differ between runs, and a pipeline that runs in a
different order each night is impossible to debug.

### Step 4 --- Prove it catches a loop

The `Acyclic` in DAG is a rule, not a decoration. Break it
deliberately.

In [ ]:
broken = {"a": ["b"], "b": ["c"], "c": ["a"]}

try:
    plan(broken)
except ValueError as problem:
    print("caught:", problem)

### Step 5 --- Run the pipeline

The runner walks the plan, calls each task, and records what
happened. If a task raises, everything after it is **skipped** ---
not attempted, not silently ignored.

In [ ]:
import time


def run_dag(dag, tasks, context=None):
    """Run every task in a safe order. Stop the line on failure."""
    context = {} if context is None else context
    results, failed = [], False

    for name in plan(dag):
        if failed:
            print(f"  SKIPPED   {name}")
            results.append({"task": name, "status": "skipped"})
            continue

        started = time.perf_counter()
        try:
            message = tasks[name](context)
            took = (time.perf_counter() - started) * 1000
            print(f"  OK        {name:<10} {message}  "
                  f"({took:.0f} ms)")
            results.append({"task": name, "status": "ok",
                            "message": message})
        except Exception as problem:
            print(f"  FAILED    {name:<10} {type(problem).__name__}"
                  f": {problem}")
            results.append({"task": name, "status": "failed",
                            "message": str(problem)})
            failed = True

    print()
    print("PIPELINE", "FAILED" if failed else "SUCCEEDED")
    return results, context


results, context = run_dag(DAG, TASKS)

### Step 6 --- Run it again, and check nothing changed

Idempotence. Run the whole thing a second time and compare the
report file. If a pipeline is safe to re-run, recovering from a
failure is trivial: fix, re-run, done.

In [ ]:
first_report = (WORK / "daily_report.txt").read_text(encoding="utf-8")

results_again, _ = run_dag(DAG, TASKS)
second_report = (WORK / "daily_report.txt").read_text(encoding="utf-8")

print()
print("first run :", first_report.strip().replace("\n", " | "))
print("second run:", second_report.strip().replace("\n", " | "))
print("identical :", first_report == second_report)

It is identical because `report` **writes** the file rather than
appending to it, and because `train` uses a fixed seed. Both are
deliberate choices, and both are the difference between a pipeline
you can re-run and one you cannot.

### Step 7 --- Watch a failure stop the line

Now feed it broken data --- a delivery with a negative distance ---
and watch `validate` refuse.

The important part is what does **not** happen: no training, no
report. A model trained on bad data would have looked perfectly
healthy.

In [ ]:
bad = pd.read_csv(DATA).copy()
bad.loc[0, "distance_km"] = -5.0
bad_path = WORK / "bad_orders.csv"
bad.to_csv(bad_path, index=False)


def extract_bad(context):
    """Same as extract, but reads the corrupted file."""
    context["orders"] = pd.read_csv(bad_path)
    return f"read {len(context['orders'])} orders"


bad_tasks = dict(TASKS, extract=extract_bad)
bad_results, _ = run_dag(DAG, bad_tasks)

Three tasks skipped. Yesterday's report is still on disk, unchanged
and still correct --- far better than a fresh report built on
rubbish.

### Step 8 --- Retry the things worth retrying

Not every failure deserves to stop the line. A network hiccup is
worth a second attempt; bad data is not.

Airflow gives every task a `retries` setting. Here is the same idea,
small enough to read.

In [ ]:
attempts_made = {"count": 0}


def flaky_download(context):
    """Fails the first two times, like a real network."""
    attempts_made["count"] += 1
    if attempts_made["count"] < 3:
        raise ConnectionError("the weather service did not answer")
    return f"downloaded on attempt {attempts_made['count']}"


def with_retries(task, tries=3, wait=0.1):
    """Wrap a task so it is retried before being given up on."""
    def wrapped(context):
        for attempt in range(1, tries + 1):
            try:
                return task(context)
            except Exception as problem:
                print(f"      attempt {attempt} failed: {problem}")
                if attempt == tries:
                    raise
                time.sleep(wait)
    return wrapped


print(with_retries(flaky_download)({}))

Retry things that might work next time. Never retry something that
failed because the data is wrong --- you will just fail three times
as slowly.

### Step 9 --- Say when it should run

Five fields: minute, hour, day of month, month, day of week. A `*`
means "every".

You will use these in Airflow, in Kubernetes CronJobs, and on
servers for the rest of your career.

In [ ]:
schedules = [
    ("0 2 * * *",   "every day at 02:00"),
    ("*/15 * * * *", "every 15 minutes"),
    ("0 9 * * 1",   "every Monday at 09:00"),
    ("0 0 1 * *",   "the first day of every month, midnight"),
    ("30 6 * * 1-5", "06:30 on weekdays only"),
]

print(f"{'cron':<15} {'means'}")
print("-" * 55)
for expression, meaning in schedules:
    print(f"{expression:<15} {meaning}")

print()
print("Our pipeline retrains overnight, so:  0 2 * * *")
print("Why 2 a.m.? Few orders, spare machines, and a human is")
print("awake by the time anyone needs the report.")

### Step 10 --- Batch or stream?

The last idea from L25. Both are pipelines; they differ in when they
run.

In [ ]:
print(f"{'':<18}{'BATCH':<28}{'STREAM'}")
print("-" * 74)
rows = [
    ("runs", "on a schedule", "continuously"),
    ("handles", "all of yesterday at once", "one event at a time"),
    ("delay", "hours", "seconds"),
    ("tools", "Airflow, cron", "Kafka, Flink"),
    ("good for", "retraining, reports", "fraud alerts, live ETA"),
    ("harder", "nothing much", "ordering, failures, cost"),
]
for label, batch, stream in rows:
    print(f"{label:<18}{batch:<28}{stream}")

print()
print("Our delivery service uses both: retrain nightly in batch,")
print("and serve predictions one request at a time in P07.")
print()
print("Choose batch unless you can say what breaks if the answer")
print("is an hour late. Streaming costs far more to run and to fix.")

---

# Your turn

The walkthrough above is finished. Now you write some code.

There are **3 tasks**. Each one is small. Each one has a hint.
Do them in order.

Where you see `# TODO`, replace that line with your own code. Do not delete
the variable name on the left of the `=` sign --- the self-check at the
bottom looks for exactly that name.

When you have tried all three, run the **self-check** cell at the end. It
prints a table telling you which tasks are correct. You can run it as many
times as you like.

### Task T1 --- Add a task to the pipeline


    The team wants a copy of the trained model saved every night, **after**
    training but **before** the report.

    1. Write a task function `save_model(context)` that writes
       `context["model"]` to `work/model.joblib` with `joblib.dump` and
       returns the string `"model saved"`.
    2. Add it to a new graph `T1_dag` so that `save_model` waits for
       `train`, and `report` waits for `save_model`.
    3. Run the pipeline with `run_dag` and keep the results in
       `T1_results`.


> **Hint.** Copy `DAG` with `dict(DAG)`, then set `T1_dag["save_model"] = ["train"]` and change `T1_dag["report"]` to `["save_model"]`. Build the task map with `dict(TASKS, save_model=save_model)` and pass both to `run_dag`. Careful: `run_dag` hands back **two** things, so unpack them --- `T1_results, _ = run_dag(...)`.

In [ ]:
import joblib


def save_model(context):
    # TODO: dump context["model"] to work/model.joblib, return "model saved"
    return None


# TODO: a new graph with save_model between train and report
T1_dag = None

# TODO: run it
T1_results = None

print(T1_results)

### Task T2 --- Predict the damage


    No code to run for this one --- think it through.

    Using the graph from Step 2 (`extract -> validate -> transform -> train
    -> report`), suppose **`transform` fails**.

    Put the names of the tasks that would be **skipped**, in run order, in
    the list `T2_skipped`. Then put the number of tasks that would report
    `ok` in `T2_ok_count`.


> **Hint.** Look at the run order. Everything before the failure has already succeeded. The failing task itself is 'failed', not 'skipped'. Everything after it is skipped.

In [ ]:
# TODO: which tasks are skipped when transform fails?
T2_skipped = None

# TODO: how many tasks report ok?
T2_ok_count = None

print("skipped:", T2_skipped, " ok:", T2_ok_count)

### Task T3 --- Read and write a schedule


    Two parts.

    1. `T3_meaning` --- in your own words, what does `15 3 * * 0` mean?
       Your answer must be a string containing the hour `3`, the minute
       `15`, and the word `Sunday`.
    2. `T3_cron` --- write the cron expression for **every day at 23:30**.


> **Hint.** The five fields are minute, hour, day-of-month, month, day-of-week. In the day-of-week field, 0 is Sunday. For part 2, two numbers and three stars.

In [ ]:
# TODO: what does "15 3 * * 0" mean, in words?
T3_meaning = None

# TODO: cron for every day at 23:30
T3_cron = None

print(T3_meaning)
print(T3_cron)

---

## Self-check

Run the cell below to mark your work.

In [ ]:
# ------------------------------------------------------------------
# SELF-CHECK --- run this when you have attempted the tasks above.
# It never breaks your notebook. A task you have not done yet simply
# shows FAIL.
# ------------------------------------------------------------------

_results = []


def _check(label, fn):
    """Evaluate one graded condition without ever raising."""
    try:
        ok = bool(fn())
    except Exception:
        ok = False
    _results.append((label, ok))


_check('T1 | save_model runs after train and before report', lambda: plan(T1_dag).index('train') < plan(T1_dag).index('save_model') < plan(T1_dag).index('report'))
_check('T1 | every task in the run succeeded', lambda: all(r['status'] == 'ok' for r in T1_results) and len(T1_results) == 6)
_check('T1 | the model file was really written', lambda: (WORK / 'model.joblib').is_file() and (WORK / 'model.joblib').stat().st_size > 100)
_check('T2 | train and report are skipped, in that order', lambda: list(T2_skipped) == ['train', 'report'])
_check('T2 | extract and validate are the two that succeeded', lambda: int(T2_ok_count) == 2)
_check('T3 | the explanation names the hour, the minute and the day', lambda: '3' in T3_meaning and '15' in T3_meaning and 'sunday' in T3_meaning.lower())
_check('T3 | T3_cron has the five cron fields', lambda: len(T3_cron.split()) == 5)
_check('T3 | it means 23:30 every day', lambda: T3_cron.split()[:2] == ['30', '23'] and T3_cron.split()[2:] == ['*', '*', '*'])

print("==================================================================")
print("SELF-CHECK   Practical 12 --- A Pipeline You Can Re-run")
print("==================================================================")
for _label, _ok in _results:
    print(f"  [{'PASS' if _ok else 'FAIL'}]  {_label}")
print("------------------------------------------------------------------")
_passed = sum(1 for _, _ok in _results if _ok)
print(f"  {_passed} of {len(_results)} checks passed")
print("==================================================================")
if _passed == len(_results):
    print("Well done. Save the notebook and submit it.")
else:
    print("Read the FAIL lines above, fix those tasks, run this cell again.")

---

## What to submit

1. This notebook, with every cell run and its output visible.
2. Your `work/daily_report.txt`.
3. In a markdown cell, one sentence saying what would have gone wrong if `validate` had not stopped the pipeline in Step 7.

Name your file `PXX_<your-roll-number>.ipynb` before you upload it.

### How this practical is marked

| What is marked | Marks |
|---|---|
| Walkthrough run end to end, pipeline succeeds then fails safely | 3 |
| Task T1 --- a new task wired into the graph | 3 |
| Task T2 --- the effect of a failure predicted correctly | 2 |
| Task T3 --- schedules read and written | 2 |
| **Total** | **10** |

---

## Read more

- Apache Airflow --- Core concepts: DAGs --- <https://airflow.apache.org/docs/apache-airflow/stable/core-concepts/dags.html>
- Apache Airflow --- Tasks and dependencies --- <https://airflow.apache.org/docs/apache-airflow/stable/core-concepts/tasks.html>
- Apache Airflow --- Cron and time intervals --- <https://airflow.apache.org/docs/apache-airflow/stable/authoring-and-scheduling/cron.html>
- Kubernetes --- CronJob --- <https://kubernetes.io/docs/concepts/workloads/controllers/cron-jobs/>